## Summary

- **Best fidelity achieved:** displayed above
- **Best hyperparameters:** listed in the "Best Parameters" section
- **To run full optimization:** set `dry_run = False` in cell 1 and re-run cell 4
- **To tune experiment parameters:** edit `N_SHOTS`, `BATCH_SIZE`, or `N_QUBITS` in `optuna_search.py`


In [ ]:
# Load and run Optuna search
import traceback, time

script_path = os.path.join(project_root, 'shadow_gym', 'notebooks', 'optuna_search.py')

try:
    # Read and load the script
    with open(script_path, 'r') as f:
        src = f.read()
    
    # Disable the script's main block so we can safely import its objective
    src = src.replace('if __name__ == "__main__":', 'if False:')
    
    # Execute script in a safe namespace
    module_ns = {'__name__': 'optuna_search_notebook_exec', '__file__': script_path}
    exec(compile(src, script_path, 'exec'), module_ns)
    
    # Extract the objective function
    objective = module_ns.get('objective')
    if objective is None:
        raise RuntimeError('optuna_search.py does not define an objective function')
    
    # Create and run the study
    study_name = f'ai_agent_tuning_{int(time.time())}'
    study = optuna.create_study(direction='maximize', study_name=study_name)
    
    n_trials = dry_trials if dry_run else full_trials
    print(f'\nRunning {n_trials} trial(s)...\n')
    study.optimize(objective, n_trials=n_trials, n_jobs=1)
    
    # Display results
    print('\n' + '='*70)
    print(f'OPTIMIZATION COMPLETE')
    print(f'Best Fidelity: {study.best_value:.6f}')
    print('='*70)
    print('Best Parameters:')
    for key, value in study.best_params.items():
        print(f'  {key}: {value}')
    print('='*70)
    
except Exception:
    print('\n✗ Error running Optuna search:')
    traceback.print_exc()


In [ ]:
# Validate required dependencies
try:
    import optuna
    from shadow_gym.src import QuantumEnvironment, ShadowProcessor, ActiveInferenceAgent
    from shadow_gym.src.utils import fidelity
    print('✓ All imports OK: optuna, shadow_gym modules')
except Exception as e:
    print(f'✗ Import failed: {e}')
    raise


In [ ]:
# Configuration: Search parameters
dry_run = True  # Set to False for full 100-trial optimization
dry_trials = 2
full_trials = 100

# Setup repository paths
import sys, os

def find_repo_root(start=os.getcwd()):
    cur = os.path.abspath(start)
    while True:
        if os.path.isdir(os.path.join(cur, 'shadow_gym')) and os.path.isfile(os.path.join(cur, 'shadow_gym', '__init__.py')):
            return cur
        parent = os.path.dirname(cur)
        if parent == cur:
            return None
        cur = parent

project_root = find_repo_root() or os.getcwd()
sys.path.insert(0, os.path.abspath(project_root))
src_path = os.path.join(project_root, 'shadow_gym', 'src')
if os.path.isdir(src_path):
    sys.path.insert(0, os.path.abspath(src_path))

print(f'Project root: {project_root}')
print(f'Src path: {src_path}')


# Optuna Hyperparameter Search

This notebook runs the Optuna hyperparameter optimization for the Active Inference quantum agent.

**Quick start:**
- Set `dry_run = True` for a quick 2-trial smoke test
- Set `dry_run = False` for the full 100-trial optimization
